In [0]:
%fs head /Volumes/workspace/distribuidora_los_andes/datos_proyecto/entrada/productos.csv

[Truncated to first 65536 bytes]
id_producto,nombre_producto,categoria,proveedor,precio_unitario_soles,stock_disponible,unidades_vendidas,ventas_soles,unidades_vendidas_90d,fecha_ultima_venta,unidades_2024,ventas_soles_2024,unidades_2025,ventas_soles_2025,unidades_2026,ventas_soles_2026,origen_dato
1,Azucar Rubia 1,Abarrotes,Distribuciones Lima SAC,4.95,67,41,202.95,1,2026-08-08,8,39.60,21,103.95,12,59.40,base_proyecto
2,Gaseosa Cola 2,Bebidas,Comercial San Martin SAC,5.00,84,43,215.00,3,2026-08-09,16,80.00,15,75.00,12,60.00,base_proyecto
3,Agua Mineral 3,Lacteos,Alimentos del Pacifico SAC,4.75,101,69,327.75,5,2026-08-10,24,114.00,25,118.75,20,95.00,base_proyecto
4,Leche Evaporada 4,Limpieza,Corporacion Andina SAC,7.80,118,55,429.00,7,2026-08-11,16,124.80,19,148.20,20,156.00,base_proyecto
5,Yogurt Natural 5,Cuidado Personal,Importaciones del Norte SAC,9.25,135,41,379.25,1,2026-08-12,8,74.00,21,194.25,12,111.00,base_proyecto
6,Detergente 6,Snacks,Productos Nacionales SAC,14.40,152,43,61

In [0]:
ruta = "/Volumes/workspace/distribuidora_los_andes/datos_proyecto/entrada/productos.csv"
dfProductos = spark.read.csv(ruta, header=True, inferSchema=True)

print("Productos cargados:", dfProductos.count())
display(dfProductos.limit(5))

Productos cargados: 50000


id_producto,nombre_producto,categoria,proveedor,precio_unitario_soles,stock_disponible,unidades_vendidas,ventas_soles,unidades_vendidas_90d,fecha_ultima_venta,unidades_2024,ventas_soles_2024,unidades_2025,ventas_soles_2025,unidades_2026,ventas_soles_2026,origen_dato
1,Azucar Rubia 1,Abarrotes,Distribuciones Lima SAC,4.95,67,41,202.95,1,2026-08-08,8,39.6,21,103.95,12,59.4,base_proyecto
2,Gaseosa Cola 2,Bebidas,Comercial San Martin SAC,5.0,84,43,215.0,3,2026-08-09,16,80.0,15,75.0,12,60.0,base_proyecto
3,Agua Mineral 3,Lacteos,Alimentos del Pacifico SAC,4.75,101,69,327.75,5,2026-08-10,24,114.0,25,118.75,20,95.0,base_proyecto
4,Leche Evaporada 4,Limpieza,Corporacion Andina SAC,7.8,118,55,429.0,7,2026-08-11,16,124.8,19,148.2,20,156.0,base_proyecto
5,Yogurt Natural 5,Cuidado Personal,Importaciones del Norte SAC,9.25,135,41,379.25,1,2026-08-12,8,74.0,21,194.25,12,111.0,base_proyecto


In [0]:
resumen_origen = dfProductos.groupBy("origen_dato").count()
display(resumen_origen.orderBy("origen_dato"))

origen_dato,count
base_proyecto,200
simulado,49800


In [0]:
medidas = dfProductos.select("precio_unitario_soles", "stock_disponible", "unidades_vendidas_90d")
display(medidas.describe())

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7570884009226945>, line 1
----> 1 medidas = dfProductos.select("precio_unitario_soles", "stock_disponible", "unidades_vendidas_90d")
      2 display(medidas.describe())

NameError: name 'dfProductos' is not defined

In [0]:
duplicados = dfProductos.groupBy("id_producto").count().filter("count > 1")
print("IDs de producto repetidos:", duplicados.count())

IDs de producto repetidos: 0


In [0]:
campos = ["id_producto", "categoria", "precio_unitario_soles", "stock_disponible"]
dfCompletos = dfProductos.na.drop(subset=campos)

print("Filas con algún campo requerido nulo:", dfProductos.count() - dfCompletos.count())
print("Filas con los cuatro campos completos:", dfCompletos.count())

Filas con algún campo requerido nulo: 0
Filas con los cuatro campos completos: 50000


In [0]:
dfValidos = dfCompletos.filter("precio_unitario_soles > 0 AND stock_disponible >= 0")

print("Productos con precio o stock inválido:", dfCompletos.count() - dfValidos.count())
print("Productos válidos:", dfValidos.count())

Productos con precio o stock inválido: 0
Productos válidos: 50000


In [0]:
from pyspark.sql import functions as F

dfInventario = dfValidos.withColumn(
    "valor_inventario_soles",
    F.col("stock_disponible") * F.col("precio_unitario_soles").cast("decimal(12,2)")
)

display(dfInventario.select("nombre_producto", "stock_disponible",
                            "precio_unitario_soles", "valor_inventario_soles").limit(5))

nombre_producto,stock_disponible,precio_unitario_soles,valor_inventario_soles
Azucar Rubia 1,67,4.95,331.65
Gaseosa Cola 2,84,5.0,420.00
Agua Mineral 3,101,4.75,479.75
Leche Evaporada 4,118,7.8,920.40
Yogurt Natural 5,135,9.25,1248.75


In [0]:
dfInventario = dfInventario.withColumn(
    "estado_stock",
    F.when(F.col("stock_disponible") < 10, "Stock bajo").otherwise("Stock suficiente")
)

display(dfInventario.groupBy("origen_dato", "estado_stock").count())

origen_dato,estado_stock,count
simulado,Stock bajo,5791
simulado,Stock suficiente,44009
base_proyecto,Stock suficiente,200


In [0]:
tabla = "workspace.distribuidora_los_andes.productos_silver"

dfInventario.write.format("delta").mode("overwrite").saveAsTable(tabla)
print("Productos guardados en la tabla:", spark.table(tabla).count())

Productos guardados en la tabla: 50000


In [0]:
%sql
SELECT categoria,
       SUM(CAST(ventas_soles AS DECIMAL(14,2))) AS ventas_totales_soles
FROM workspace.distribuidora_los_andes.productos_silver
WHERE origen_dato = 'base_proyecto'
GROUP BY categoria
ORDER BY ventas_totales_soles DESC

categoria,ventas_totales_soles
Abarrotes,9889.35
Snacks,8795.65
Hogar,8767.25
Limpieza,8646.55
Congelados,8612.00
Papeleria,7854.65
Cuidado Personal,7844.75
Panificacion,7689.75
Aceites y Salsas,7588.05
Bebidas,5374.55


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT nombre_producto, stock_disponible, unidades_vendidas_90d
FROM workspace.distribuidora_los_andes.productos_silver
WHERE origen_dato = 'simulado' AND estado_stock = 'Stock bajo'
ORDER BY unidades_vendidas_90d DESC, stock_disponible ASC, id_producto
LIMIT 10

nombre_producto,stock_disponible,unidades_vendidas_90d
Arroz extra Sierra Brisa 1.5 kg - SKU 42976,6,38
Arroz extra Sierra Brisa 1.5 kg - SKU 39013,0,37
Arroz extra Sierra Brisa 1.5 kg - SKU 02795,0,36
Arroz extra Sierra Brisa 1.5 kg - SKU 36292,0,36
Arroz extra Sierra Brisa 1.5 kg - SKU 32280,8,36
Arroz extra Sierra Brisa 1.5 kg - SKU 19559,0,35
Arroz extra Sierra Brisa 1.5 kg - SKU 17762,0,34
Arroz extra Sierra Brisa 1.5 kg - SKU 25817,9,34
Arroz extra Sierra Brisa 1.5 kg - SKU 28779,8,33
Aceite vegetal Valle Violeta 500 ml - SKU 29482,0,32


Databricks visualization. Run in Databricks to view.

In [0]:
import time

consulta = """
SELECT nombre_producto, stock_disponible, unidades_vendidas_90d
FROM workspace.distribuidora_los_andes.productos_silver
WHERE origen_dato = 'simulado'
  AND estado_stock = 'Stock bajo'
ORDER BY unidades_vendidas_90d DESC,
         stock_disponible ASC,
         id_producto
LIMIT 10
"""

inicio = time.perf_counter()

resultado = spark.sql(consulta).collect()

fin = time.perf_counter()

tiempo = fin - inicio

print("Productos encontrados:", len(resultado))
print("Tiempo de ejecución:", round(tiempo, 4), "segundos")

display(resultado)

Productos encontrados: 10
Tiempo de ejecución: 16.7759 segundos


nombre_producto,stock_disponible,unidades_vendidas_90d
Arroz extra Sierra Brisa 1.5 kg - SKU 42976,6,38
Arroz extra Sierra Brisa 1.5 kg - SKU 39013,0,37
Arroz extra Sierra Brisa 1.5 kg - SKU 02795,0,36
Arroz extra Sierra Brisa 1.5 kg - SKU 36292,0,36
Arroz extra Sierra Brisa 1.5 kg - SKU 32280,8,36
Arroz extra Sierra Brisa 1.5 kg - SKU 19559,0,35
Arroz extra Sierra Brisa 1.5 kg - SKU 17762,0,34
Arroz extra Sierra Brisa 1.5 kg - SKU 25817,9,34
Arroz extra Sierra Brisa 1.5 kg - SKU 28779,8,33
Aceite vegetal Valle Violeta 500 ml - SKU 29482,0,32


In [0]:
from pyspark.sql.functions import count
import time

inicio = time.perf_counter()

resultado = dfProductos.groupBy(
    "categoria",
    "origen_dato"
).agg(
    count("*").alias("total_productos")
).orderBy(
    "total_productos",
    ascending=False
)

datos = resultado.collect()

fin = time.perf_counter()

print("Total de grupos:", len(datos))
print("Tiempo de procesamiento:", round(fin - inicio, 4), "segundos")

display(resultado)

Total de grupos: 30
Tiempo de procesamiento: 0.7725 segundos


categoria,origen_dato,total_productos
Lacteos,simulado,5077
Aceites y Salsas,simulado,5057
Bebidas,simulado,4954
Snacks,simulado,4949
Granos y Cereales,simulado,4911
Bebes,simulado,2526
Conservas,simulado,2520
Cuidado Personal,simulado,2518
Hogar,simulado,2518
Mascotas,simulado,2485


In [0]:
import time
from pyspark.sql.functions import count

# Cantidad total de productos
total_productos = dfProductos.count()

# Medir el procesamiento
inicio = time.perf_counter()

resultado = dfProductos.groupBy(
    "categoria", "origen_dato"
).agg(
    count("*").alias("total_productos")
)

datos = resultado.collect()

fin = time.perf_counter()

tiempo = fin - inicio

# Calcular throughput
throughput = total_productos / tiempo

print("Total de productos:", total_productos)
print("Tiempo de procesamiento:", round(tiempo, 4), "segundos")
print("Throughput:", round(throughput, 2), "registros/segundo")

Total de productos: 50000
Tiempo de procesamiento: 0.5959 segundos
Throughput: 83900.66 registros/segundo


In [0]:
import time
from pyspark.sql.functions import count

# Seleccionar 2500 productos
dfComparacion = dfProductos.limit(2500)

# Contar los registros
total = dfComparacion.count()

# Medir el procesamiento
inicio = time.perf_counter()

resultado = dfComparacion.groupBy(
    "categoria"
).agg(
    count("*").alias("total_productos")
)

datos = resultado.collect()

fin = time.perf_counter()

tiempo = fin - inicio

# Calcular throughput
throughput = total / tiempo

print("Total de productos:", total)
print("Total de grupos:", len(datos))
print("Tiempo de procesamiento:", round(tiempo, 6), "segundos")
print("Throughput:", round(throughput, 2), "registros/segundo")

display(resultado)

Total de productos: 2500
Total de grupos: 15
Tiempo de procesamiento: 0.559899 segundos
Throughput: 4465.09 registros/segundo


categoria,total_productos
Abarrotes,126
Bebidas,248
Lacteos,247
Limpieza,135
Cuidado Personal,130
Snacks,244
Conservas,131
Granos y Cereales,241
Aceites y Salsas,233
Panificacion,130


In [0]:

# EXP-BI-05 - BIG DATA
# Carga de productos desde Unity Catalog Volume

ruta = "/Volumes/workspace/distribuidora_los_andes/datos_proyecto/entrada/productos.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(ruta)

# Mostrar los datos
display(df)


id_producto,nombre_producto,categoria,proveedor,precio_unitario_soles,stock_disponible,unidades_vendidas,ventas_soles,unidades_vendidas_90d,fecha_ultima_venta,unidades_2024,ventas_soles_2024,unidades_2025,ventas_soles_2025,unidades_2026,ventas_soles_2026,origen_dato
1,Azucar Rubia 1,Abarrotes,Distribuciones Lima SAC,4.95,67,41,202.95,1,2026-08-08,8,39.6,21,103.95,12,59.4,base_proyecto
2,Gaseosa Cola 2,Bebidas,Comercial San Martin SAC,5.0,84,43,215.0,3,2026-08-09,16,80.0,15,75.0,12,60.0,base_proyecto
3,Agua Mineral 3,Lacteos,Alimentos del Pacifico SAC,4.75,101,69,327.75,5,2026-08-10,24,114.0,25,118.75,20,95.0,base_proyecto
4,Leche Evaporada 4,Limpieza,Corporacion Andina SAC,7.8,118,55,429.0,7,2026-08-11,16,124.8,19,148.2,20,156.0,base_proyecto
5,Yogurt Natural 5,Cuidado Personal,Importaciones del Norte SAC,9.25,135,41,379.25,1,2026-08-12,8,74.0,21,194.25,12,111.0,base_proyecto
6,Detergente 6,Snacks,Productos Nacionales SAC,14.4,152,43,619.2,3,2026-08-13,16,230.4,15,216.0,12,172.8,base_proyecto
7,Jabon de Tocador 7,Conservas,Comercializadora Central SAC,3.8,169,69,262.2,5,2026-08-14,24,91.2,25,95.0,20,76.0,base_proyecto
8,Galletas 8,Granos y Cereales,Inversiones Pacifico SAC,3.25,186,55,178.75,7,2026-08-15,16,52.0,19,61.75,20,65.0,base_proyecto
9,Chocolate 9,Aceites y Salsas,Distribuidora Santa Rosa SAC,6.4,203,41,262.4,1,2026-08-16,8,51.2,21,134.4,12,76.8,base_proyecto
10,Atun en Conserva 10,Panificacion,Comercial Los Olivos SAC,8.75,220,43,376.25,3,2026-08-17,16,140.0,15,131.25,12,105.0,base_proyecto


In [0]:

# Revisar columnas y tipos de datos
df.printSchema()

# Cantidad de registros
print("Total de productos:", df.count())

# Mostrar los primeros registros
display(df.limit(10))


root
 |-- id_producto: integer (nullable = true)
 |-- nombre_producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- proveedor: string (nullable = true)
 |-- precio_unitario_soles: double (nullable = true)
 |-- stock_disponible: integer (nullable = true)
 |-- unidades_vendidas: integer (nullable = true)
 |-- ventas_soles: double (nullable = true)
 |-- unidades_vendidas_90d: integer (nullable = true)
 |-- fecha_ultima_venta: date (nullable = true)
 |-- unidades_2024: integer (nullable = true)
 |-- ventas_soles_2024: double (nullable = true)
 |-- unidades_2025: integer (nullable = true)
 |-- ventas_soles_2025: double (nullable = true)
 |-- unidades_2026: integer (nullable = true)
 |-- ventas_soles_2026: double (nullable = true)
 |-- origen_dato: string (nullable = true)

Total de productos: 50000


id_producto,nombre_producto,categoria,proveedor,precio_unitario_soles,stock_disponible,unidades_vendidas,ventas_soles,unidades_vendidas_90d,fecha_ultima_venta,unidades_2024,ventas_soles_2024,unidades_2025,ventas_soles_2025,unidades_2026,ventas_soles_2026,origen_dato
1,Azucar Rubia 1,Abarrotes,Distribuciones Lima SAC,4.95,67,41,202.95,1,2026-08-08,8,39.6,21,103.95,12,59.4,base_proyecto
2,Gaseosa Cola 2,Bebidas,Comercial San Martin SAC,5.0,84,43,215.0,3,2026-08-09,16,80.0,15,75.0,12,60.0,base_proyecto
3,Agua Mineral 3,Lacteos,Alimentos del Pacifico SAC,4.75,101,69,327.75,5,2026-08-10,24,114.0,25,118.75,20,95.0,base_proyecto
4,Leche Evaporada 4,Limpieza,Corporacion Andina SAC,7.8,118,55,429.0,7,2026-08-11,16,124.8,19,148.2,20,156.0,base_proyecto
5,Yogurt Natural 5,Cuidado Personal,Importaciones del Norte SAC,9.25,135,41,379.25,1,2026-08-12,8,74.0,21,194.25,12,111.0,base_proyecto
6,Detergente 6,Snacks,Productos Nacionales SAC,14.4,152,43,619.2,3,2026-08-13,16,230.4,15,216.0,12,172.8,base_proyecto
7,Jabon de Tocador 7,Conservas,Comercializadora Central SAC,3.8,169,69,262.2,5,2026-08-14,24,91.2,25,95.0,20,76.0,base_proyecto
8,Galletas 8,Granos y Cereales,Inversiones Pacifico SAC,3.25,186,55,178.75,7,2026-08-15,16,52.0,19,61.75,20,65.0,base_proyecto
9,Chocolate 9,Aceites y Salsas,Distribuidora Santa Rosa SAC,6.4,203,41,262.4,1,2026-08-16,8,51.2,21,134.4,12,76.8,base_proyecto
10,Atun en Conserva 10,Panificacion,Comercial Los Olivos SAC,8.75,220,43,376.25,3,2026-08-17,16,140.0,15,131.25,12,105.0,base_proyecto


In [0]:

# EXP-BI-05 - BIG DATA
# Validar las columnas del CSV

columnas_necesarias = [
    "id_producto",
    "nombre_producto",
    "precio_unitario_soles",
    "stock_disponible"
]

for columna in columnas_necesarias:
    if columna in df.columns:
        print(f"OK: {columna}")
    else:
        raise Exception(f"FALTA: {columna}")

print("\nValidación completada correctamente")
print("Total de productos:", df.count())



OK: id_producto
OK: nombre_producto
OK: precio_unitario_soles
OK: stock_disponible

Validación completada correctamente
Total de productos: 50000


In [0]:

from pyspark.sql import functions as F
from time import perf_counter

umbrales = [50, 100, 150]
resultados = []

for umbral in umbrales:

    inicio = perf_counter()
    productos_alerta = df.filter(
        F.col("stock_disponible") < umbral
    )
    resultado = productos_alerta.agg(
        F.count("*").alias("productos_alerta"),
        F.sum(
            F.col("stock_disponible") *
            F.col("precio_unitario_soles")
        ).alias("valor_inventario_riesgo")
    ).collect()[0]

    fin = perf_counter()

    tiempo_ms = (fin - inicio) * 1000

    resultados.append((
        umbral,
        resultado["productos_alerta"],
        float(resultado["valor_inventario_riesgo"] or 0),
        round(tiempo_ms, 3)
    ))
df_resultados = spark.createDataFrame(
    resultados,
    [
        "umbral_stock",
        "productos_en_alerta",
        "valor_inventario_riesgo",
        "tiempo_ms"
    ]
)

display(df_resultados.orderBy("umbral_stock"))


umbral_stock,productos_en_alerta,valor_inventario_riesgo,tiempo_ms
50,14766,9820822.040000003,671.326
100,21506,2.792071289E7,527.468
150,28372,5.957074927999998E7,452.564
